**CI twin of `ch13-svm.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import numpy as np

df = load_csv("penguins")
two = df[df["species"].isin(["Adelie", "Gentoo"])].dropna(
    subset=["flipper_length_mm", "bill_length_mm"])
y = (two["species"] == "Gentoo").astype(int)
X = two[["flipper_length_mm", "bill_length_mm"]]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

svm = SVC(kernel="linear", C=1.0).fit(Xtr_s, ytr)
print(f"training birds: {len(Xtr_s)}   support vectors: {len(svm.support_)}")

# Draw boundary (decision_function = 0) and curbs (= ±1).
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
zz = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(5.2, 4))
ax.scatter(Xtr_s[:, 0], Xtr_s[:, 1], c=ytr, cmap="coolwarm", s=12, alpha=0.6)
ax.contour(xx, yy, zz, levels=[-1, 0, 1],
           linestyles=["--", "-", "--"], colors="k", linewidths=1)
sv = svm.support_vectors_
ax.scatter(sv[:, 0], sv[:, 1], s=90, facecolors="none",
           edgecolors="k", label="support vectors")
ax.set_xlabel("flipper length (scaled)")
ax.set_ylabel("bill length (scaled)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
def boundary_shift(remove_idx):
    mask = np.ones(len(Xtr_s), dtype=bool)
    mask[remove_idx] = False
    refit = SVC(kernel="linear", C=1.0).fit(Xtr_s[mask], ytr.values[mask])
    return float(np.linalg.norm(svm.coef_[0] - refit.coef_[0]))

deep_bird = int(np.argmax(np.abs(svm.decision_function(Xtr_s))))
curb_bird = int(svm.support_[0])

print(f"removing the deepest bird:  boundary moved {boundary_shift(deep_bird):.6f}")
print(f"removing one curb bird:     boundary moved {boundary_shift(curb_bird):.6f}")

In [ ]:
from sklearn.metrics import accuracy_score

print("   C      support vectors   street width   test acc")
for C in (0.01, 0.1, 1.0, 10.0, 100.0):
    m = SVC(kernel="linear", C=C).fit(Xtr_s, ytr)
    width = 2 / np.linalg.norm(m.coef_[0])
    acc = accuracy_score(yte, m.predict(Xte_s))
    print(f"{C:7}        {len(m.support_):3}            {width:.3f}        {acc:.3f}")

In [ ]:
from sklearn.datasets import make_circles

Xc, yc = make_circles(n_samples=300, factor=0.4, noise=0.08, random_state=0)
Ctr, Cte, ctr, cte = train_test_split(Xc, yc, test_size=0.25,
                                      random_state=42, stratify=yc)

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.4))
for ax, kernel in zip(axes, ("linear", "rbf")):
    m = SVC(kernel=kernel).fit(Ctr, ctr)
    acc = accuracy_score(cte, m.predict(Cte))
    gx, gy = np.meshgrid(np.linspace(-1.4, 1.4, 200),
                         np.linspace(-1.4, 1.4, 200))
    gz = m.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, gz > 0, alpha=0.15, cmap="coolwarm")
    ax.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", s=8)
    ax.set_title(f"{kernel} kernel — acc {acc:.3f}", fontsize=9)
plt.show()

In [ ]:
model = SVC(kernel="linear", C=1.0)
model.fit(Xtr_s, ytr)

run_tests([
    ("held-out accuracy", round(
        accuracy_score(yte, model.predict(Xte_s)), 3), 1.0),
    ("birds holding up the street", len(model.support_), 13),
])

In [ ]:
import math

def street_side(w, b, p):
    score = sum(wi * pi for wi, pi in zip(w, p)) + b
    return 1 if score >= 0 else -1

def margin_width(w):
    return 2 / math.sqrt(sum(wi ** 2 for wi in w))

run_tests([
    ("north side", street_side([2.0, -1.0], 0.5, [1.0, 1.0]), 1),
    ("south side", street_side([2.0, -1.0], 0.5, [-1.0, 1.0]), -1),
    ("standing on the line counts as +1",
     street_side([1.0, 0.0], -2.0, [2.0, 5.0]), 1),
    ("3-4-5 street", margin_width([3.0, 4.0]), 0.4),
    ("small weights, wide street", margin_width([0.5, 0.0]), 4.0),
], tol=1e-9)